# Monte Carlo Options Pricing - Basic Pricing

This notebook demonstrates basic option pricing using Monte Carlo simulation.

## Topics Covered:
1. European Option Pricing
2. Asian Option Pricing
3. Comparison with Black-Scholes
4. Convergence Analysis

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../src')

from mc_pricing import (
    EuropeanOption, AsianOption, BlackScholes,
    MonteCarloSimulator, Visualizer
)

%matplotlib inline
plt.style.use('seaborn-v0_8')

## 1. European Call Option

In [ ]:
# Define option parameters
option = EuropeanOption(
    option_type='call',
    strike=100.0,
    maturity=1.0,
    spot=100.0,
    rate=0.05,
    volatility=0.2
)

print(option)

In [ ]:
# Price using Black-Scholes (analytical)
bs_price = BlackScholes.price(
    option_type='call',
    spot=100.0,
    strike=100.0,
    maturity=1.0,
    rate=0.05,
    volatility=0.2
)

print(f"Black-Scholes Price: ${bs_price:.4f}")

In [ ]:
# Price using Monte Carlo
simulator = MonteCarloSimulator(n_simulations=100000, seed=42)
mc_price, mc_error = simulator.price(option)

print(f"Monte Carlo Price: ${mc_price:.4f} ± ${mc_error:.4f}")
print(f"Difference: ${abs(mc_price - bs_price):.4f}")
print(f"Within 2 std errors: {abs(mc_price - bs_price) < 2 * mc_error}")

## 2. Convergence Analysis

In [ ]:
# Test convergence for different sample sizes
n_trials = np.array([1000, 5000, 10000, 50000, 100000, 500000])

simulator = MonteCarloSimulator(n_simulations=10000, seed=42)
prices, std_errors = simulator.convergence_test(option, n_trials)

# Plot convergence
viz = Visualizer()
viz.plot_convergence(
    n_trials,
    prices,
    std_errors,
    true_price=bs_price,
    title="Monte Carlo Convergence for European Call"
)

## 3. Asian Option Pricing

In [ ]:
# Create Asian call option
asian_option = AsianOption(
    option_type='call',
    strike=100.0,
    maturity=1.0,
    spot=100.0,
    rate=0.05,
    volatility=0.2,
    averaging_type='arithmetic'
)

print(asian_option)

In [ ]:
# Price Asian option
asian_price, asian_error = simulator.price(asian_option)

print(f"Asian Call Price: ${asian_price:.4f} ± ${asian_error:.4f}")
print(f"European Call Price: ${mc_price:.4f}")
print(f"\nAsian call is cheaper due to reduced volatility from averaging")

## 4. Compare Call vs Put

In [ ]:
# Create put option
put_option = EuropeanOption(
    option_type='put',
    strike=100.0,
    maturity=1.0,
    spot=100.0,
    rate=0.05,
    volatility=0.2
)

# Price both
call_price, _ = simulator.price(option)
put_price, _ = simulator.price(put_option)

# Black-Scholes prices
bs_call = BlackScholes.price('call', 100, 100, 1, 0.05, 0.2)
bs_put = BlackScholes.price('put', 100, 100, 1, 0.05, 0.2)

print("Monte Carlo Prices:")
print(f"  Call: ${call_price:.4f}")
print(f"  Put:  ${put_price:.4f}")
print(f"\nBlack-Scholes Prices:")
print(f"  Call: ${bs_call:.4f}")
print(f"  Put:  ${bs_put:.4f}")

# Check put-call parity
parity_mc = call_price - put_price
parity_expected = 100 - 100 * np.exp(-0.05 * 1)
print(f"\nPut-Call Parity Check:")
print(f"  C - P = ${parity_mc:.4f}")
print(f"  S - K*exp(-rT) = ${parity_expected:.4f}")

## 5. Visualize Price Paths

In [ ]:
# Generate and visualize some sample paths
from mc_pricing.monte_carlo import PathGenerator

path_gen = PathGenerator(seed=42)
paths = path_gen.generate_paths(
    spot=100.0,
    maturity=1.0,
    rate=0.05,
    volatility=0.2,
    n_paths=1000,
    n_steps=252
)

viz.plot_price_paths(
    paths,
    n_paths_to_plot=50,
    strike=100.0,
    title="Sample Price Paths for GBM"
)

## Summary

In this notebook, we:
1. Priced European and Asian options using Monte Carlo
2. Compared MC prices with Black-Scholes analytical solutions
3. Analyzed convergence properties (O(1/√n))
4. Verified put-call parity
5. Visualized sample price paths

Next: Check out `02_variance_reduction.ipynb` for variance reduction techniques!